In [10]:
import requests
import pandas as pd

In [11]:
# Census Data Getting and Cleaning (Housing Data)

Census_API_KEY = "407372f970401ad76187cf86e251a969bb40965d"

Census_YEAR = 2022  #here can change year

Census_BASE_URL = f"https://api.census.gov/data/{Census_YEAR}/acs/acs5"

params = {
    "get": "NAME,B25077_001E,B19013_001E,B01003_001E",
    "for": "county:*",
    "key": Census_API_KEY
}

response = requests.get(Census_BASE_URL, params=params)

print("Status Code:", response.status_code)

if response.status_code != 200:
    print("API Error:")
    print(response.text)
    raise Exception("API request failed")

# =========================
# Transform to  DataFrame
# =========================

data = response.json()

df_raw = pd.DataFrame(data[1:], columns=data[0])

# raw data
df_raw.to_csv(f"census_raw_{YEAR}.csv", index=False)
print("Raw data saved.")

# =========================
# Data Cleaning
# =========================

# rename columns
df = df_raw.rename(columns={
    "NAME": "county_name",
    "B25077_001E": "median_home_value",
    "B19013_001E": "median_income",
    "B01003_001E": "population",
    "state": "state_fips",
    "county": "county_fips"
})

# change numeric
df["median_home_value"] = pd.to_numeric(df["median_home_value"], errors="coerce")
df["median_income"] = pd.to_numeric(df["median_income"], errors="coerce")
df["population"] = pd.to_numeric(df["population"], errors="coerce")

df["county_fips_full"] = df["state_fips"] + df["county_fips"]

# dropna
df = df.dropna(subset=["median_home_value", "median_income", "population"])


# price to income ratio
df["price_to_income_ratio"] = df["median_home_value"] / df["median_income"]

df.to_csv(f"census_cleaned_{YEAR}.csv", index=False)

print("Cleaned data saved successfully!")

print("\nFinal dataset preview:")
print(df.head())

Status Code: 200
Raw data saved.
Cleaned data saved successfully!

Final dataset preview:
               county_name  median_home_value  median_income  population  \
0  Autauga County, Alabama             191800          68315       58761   
1  Baldwin County, Alabama             266000          71039      233420   
2  Barbour County, Alabama             102700          39712       24877   
3     Bibb County, Alabama             120100          50669       22251   
4   Blount County, Alabama             159800          57440       59077   

  state_fips county_fips county_fips_full  price_to_income_ratio  
0         01         001            01001               2.807583  
1         01         003            01003               3.744422  
2         01         005            01005               2.586120  
3         01         007            01007               2.370286  
4         01         009            01009               2.782033  


In [19]:
#EPA Data Getting and Cleaning (PM2.5)

import requests
import pandas as pd
import time

EPA_EMAIL = "y_lee49@u.pacific.edu"
EPA_KEY = "orangeram71"
YEAR = 2022

BASE_URL = "https://aqs.epa.gov/data/api/annualData/byState"

# All US state FIPS codes
STATE_FIPS = [
    "01","02","04","05","06","08","09","10","11","12",
    "13","15","16","17","18","19","20","21","22","23",
    "24","25","26","27","28","29","30","31","32","33",
    "34","35","36","37","38","39","40","41","42","44",
    "45","46","47","48","49","50","51","53","54","55","56"
]

all_data = []

# =====================================
# Fetch Data State by State
# =====================================

for state in STATE_FIPS:

    print(f"Fetching state {state}")

    params = {
        "email": EPA_EMAIL,
        "key": EPA_KEY,
        "param": "88101",  # PM2.5
        "bdate": f"{YEAR}0101",
        "edate": f"{YEAR}1231",
        "state": state
    }

    response = requests.get(BASE_URL, params=params)

    if response.status_code != 200:
        print(f"Failed for state {state}")
        continue

    json_data = response.json()

    if "Data" not in json_data:
        continue

    df_state = pd.DataFrame(json_data["Data"])
    all_data.append(df_state)

    time.sleep(0.3)  # avoid rate limit

# =====================================
# Combine All States
# =====================================

df_pm25 = pd.concat(all_data, ignore_index=True)

# Keep relevant columns
df_pm25 = df_pm25[[
    "state_code",
    "county_code",
    "arithmetic_mean"
]]

# Convert to numeric
df_pm25["arithmetic_mean"] = pd.to_numeric(
    df_pm25["arithmetic_mean"], errors="coerce"
)

df_pm25 = df_pm25.dropna(subset=["arithmetic_mean"])

# =====================================
# Compute State Average
# =====================================

df_state_pm25 = (
    df_pm25
    .groupby("state_code", as_index=False)["arithmetic_mean"]
    .mean()
)

df_state_pm25 = df_state_pm25.rename(columns={
    "state_code": "state_fips",
    "arithmetic_mean": "pm25_state_mean"
})


df_state_pm25.to_csv(f"pm25_state_level_{YEAR}.csv", index=False)

print("State-level PM2.5 dataset saved successfully.")
print(df_state_pm25.head())

Fetching state 01
Fetching state 02
Fetching state 04
Fetching state 05
Fetching state 06
Fetching state 08
Fetching state 09
Fetching state 10
Fetching state 11
Fetching state 12
Fetching state 13
Fetching state 15
Fetching state 16
Fetching state 17
Fetching state 18
Fetching state 19
Fetching state 20
Fetching state 21
Fetching state 22
Fetching state 23
Fetching state 24
Fetching state 25
Fetching state 26
Fetching state 27
Fetching state 28
Fetching state 29
Fetching state 30
Fetching state 31
Fetching state 32
Fetching state 33
Fetching state 34
Fetching state 35
Fetching state 36
Fetching state 37
Fetching state 38
Fetching state 39
Fetching state 40
Fetching state 41
Fetching state 42
Fetching state 44
Fetching state 45
Fetching state 46
Fetching state 47
Fetching state 48
Fetching state 49
Fetching state 50
Fetching state 51
Fetching state 53
Fetching state 54
Fetching state 55
Fetching state 56
State-level PM2.5 dataset saved successfully.
  state_fips  pm25_state_mean
0     

In [31]:
# FEMA Data Getting and Cleaning (Flood, Hurricane, Fire, Extreme Temperature)

START_YEAR = 2018
END_YEAR = 2022

BASE_URL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"

CLIMATE_TYPES = [
    "Hurricane",
    "Severe Storm",
    "Flood",
    "Fire",
    "Coastal Storm",
    "Tropical Storm",
    "Tornado"
]

all_data = []

# ==========================================
# Fetch FEMA Data with Pagination
# ==========================================

for year in range(START_YEAR, END_YEAR + 1):

    print(f"\nFetching FEMA data for {year}")

    skip = 0
    page_size = 1000

    while True:

        filter_query = (
            f"declarationDate ge '{year}-01-01T00:00:00.000Z' "
            f"and declarationDate le '{year}-12-31T23:59:59.999Z'"
        )

        params = {
            "$filter": filter_query,
            "$top": page_size,
            "$skip": skip
        }

        response = requests.get(BASE_URL, params=params)

        if response.status_code != 200:
            print(f"Failed request for year {year}")
            break

        data = response.json()

        if "DisasterDeclarationsSummaries" not in data:
            break

        df_page = pd.DataFrame(data["DisasterDeclarationsSummaries"])

        if df_page.empty:
            break

        all_data.append(df_page)

        skip += page_size
        print(f"Collected {skip} rows for {year}")

    print(f"Finished year {year}")

if len(all_data) == 0:
    raise ValueError("No FEMA data collected.")

# ==========================================
# Combine All Years
# ==========================================

df_disaster = pd.concat(all_data, ignore_index=True)

# Keep only climate-related disaster types
df_disaster = df_disaster[df_disaster["incidentType"].isin(CLIMATE_TYPES)]

# ==========================================
# Create County FIPS
# ==========================================

df_disaster["state_fips"] = df_disaster["fipsStateCode"].astype(str).str.zfill(2)
df_disaster["county_fips"] = df_disaster["fipsCountyCode"].astype(str).str.zfill(3)

df_disaster["county_fips_full"] = (
    df_disaster["state_fips"] +
    df_disaster["county_fips"]
)

# Remove rows without valid county
df_disaster = df_disaster[df_disaster["county_fips"] != "000"]

# ==========================================
# Aggregate Individual Disaster Types
# ==========================================

def count_by_type(df, disaster_type, column_name):
    return (
        df[df["incidentType"] == disaster_type]
        .groupby("county_fips_full")
        .size()
        .reset_index(name=column_name)
    )

df_hurricane = count_by_type(df_disaster, "Hurricane", "hurricane_count_5yr")
df_storm = count_by_type(df_disaster, "Severe Storm", "severe_storm_count_5yr")
df_flood = count_by_type(df_disaster, "Flood", "flood_count_5yr")
df_fire = count_by_type(df_disaster, "Fire", "fire_count_5yr")
df_coastal = count_by_type(df_disaster, "Coastal Storm", "coastal_storm_count_5yr")
df_tropical = count_by_type(df_disaster, "Tropical Storm", "tropical_storm_count_5yr")
df_tornado = count_by_type(df_disaster, "Tornado", "tornado_count_5yr")

# ==========================================
# Merge All Disaster Counts
# ==========================================

dfs = [
    df_hurricane,
    df_storm,
    df_flood,
    df_fire,
    df_coastal,
    df_tropical,
    df_tornado
]

df_climate = dfs[0]

for df in dfs[1:]:
    df_climate = df_climate.merge(df, on="county_fips_full", how="outer")

df_climate = df_climate.fillna(0)

# ==========================================
# Create Total Climate Event Count
# ==========================================

climate_columns = [
    "hurricane_count_5yr",
    "severe_storm_count_5yr",
    "flood_count_5yr",
    "fire_count_5yr",
    "coastal_storm_count_5yr",
    "tropical_storm_count_5yr",
    "tornado_count_5yr"
]

df_climate["climate_event_count_5yr"] = df_climate[climate_columns].sum(axis=1)


df_climate.to_csv("fema_climate_complete_5yr.csv", index=False)

print("\nClimate dataset saved successfully.")
print(df_climate.head())


Fetching FEMA data for 2018
Collected 1000 rows for 2018
Collected 2000 rows for 2018
Finished year 2018

Fetching FEMA data for 2019
Collected 1000 rows for 2019
Collected 2000 rows for 2019
Finished year 2019

Fetching FEMA data for 2020
Collected 1000 rows for 2020
Collected 2000 rows for 2020
Collected 3000 rows for 2020
Collected 4000 rows for 2020
Collected 5000 rows for 2020
Collected 6000 rows for 2020
Collected 7000 rows for 2020
Collected 8000 rows for 2020
Collected 9000 rows for 2020
Collected 10000 rows for 2020
Finished year 2020

Fetching FEMA data for 2021
Collected 1000 rows for 2021
Collected 2000 rows for 2021
Finished year 2021

Fetching FEMA data for 2022
Collected 1000 rows for 2022
Collected 2000 rows for 2022
Finished year 2022

Climate dataset saved successfully.
  county_fips_full  hurricane_count_5yr  severe_storm_count_5yr  \
0            01001                  2.0                     0.0   
1            01003                  3.0                     0.0   